# 03 — Preprocesamiento y Limpieza
**Fase 2 · CRISP-DM** | Proyecto: Predicción de Stroke

UNAM · Facultad de Ingeniería · Minería de Datos · Semestre 2027-1

---

Operaciones aplicadas:
1. Eliminación de `id` (sin valor predictivo)
2. Eliminación del único registro con `gender='Other'`
3. Imputación de `bmi` con la **mediana**
4. `smoking_status='Unknown'` se conserva como categoría propia
5. **One-hot encoding** de variables categóricas
6. **StandardScaler** sobre variables numéricas
7. **SMOTE** para balancear clases en el conjunto de entrenamiento

Salida: `data/processed_train.csv` y `data/processed_test.csv`

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

DATA_DIR     = os.path.join(os.getcwd(), "data")
CSV_PATH     = os.path.join(DATA_DIR, "healthcare-dataset-stroke-data.csv")
RANDOM_STATE = 42
TEST_SIZE    = 0.20

df = pd.read_csv(CSV_PATH)
print(f"Dataset original: {df.shape}")

## 3.1 Limpieza básica

In [ ]:
# Eliminar columna id
df = df.drop(columns=["id"])

# Eliminar registro con gender='Other' (n=1)
df = df[df["gender"] != "Other"].reset_index(drop=True)

# Imputar bmi con la mediana
bmi_median = df["bmi"].median()
df["bmi"] = df["bmi"].fillna(bmi_median)
print(f"bmi imputado con mediana = {bmi_median:.2f}")
print(f"Dataset tras limpieza: {df.shape}")

# Verificar que no quedan nulos
assert df.isnull().sum().sum() == 0, "Aún hay valores faltantes"
print("Sin valores faltantes ✓")

## 3.2 One-hot encoding

In [ ]:
cat_cols = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]
df_enc = pd.get_dummies(df, columns=cat_cols, drop_first=False)
print(f"Columnas tras encoding: {df_enc.shape[1]}")
df_enc.head(2)

## 3.3 División train / test (80 / 20 estratificado)

In [ ]:
X = df_enc.drop(columns=["stroke"])
y = df_enc["stroke"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras")
print(f"Distribución y_train: {y_train.value_counts().to_dict()}")

## 3.4 Estandarización (StandardScaler)

In [ ]:
num_cols = ["age", "avg_glucose_level", "bmi"]
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])
print("Estandarización aplicada ✓")

## 3.5 SMOTE — balanceo de clases en entrenamiento

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Distribución y_train ANTES de SMOTE: {y_train.value_counts().to_dict()}")
print(f"Distribución y_train DESPUÉS de SMOTE: {pd.Series(y_train_res).value_counts().to_dict()}")

## 3.6 Guardar conjuntos procesados

In [ ]:
train_df = X_train_res.copy()
train_df["stroke"] = y_train_res
test_df  = X_test.copy()
test_df["stroke"]  = y_test.values

train_path = os.path.join(DATA_DIR, "processed_train.csv")
test_path  = os.path.join(DATA_DIR, "processed_test.csv")
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path,   index=False)
print(f"Guardado: {train_path}")
print(f"Guardado: {test_path}")

---
✅ **Fase de preprocesamiento completada.**  
Continúa en `04_modeling.ipynb`.